# Inherit an existing status type

Create a subtype without declaring replacement statuses, and an axis that narrows the type.

## Prepare the model

Run this cell first. It imports the required types and defines the model used by this example. Start with a fresh repository Python kernel so declarations from other examples do not enter this model.

In [1]:
from aoa.action_machine.domain import BaseStateEntity, Lifecycle, value


class OrderStateEntity(BaseStateEntity):
    """Describe one possible order status."""

    DRAFT = value(code="draft", display_name="Draft")
    PAID = value(code="paid", display_name="Payment received")


class OrderLifecycle(Lifecycle[OrderStateEntity]):
    """Allow a draft order to become paid."""

    _template = (
        Lifecycle()
        .state(OrderStateEntity.DRAFT).to(OrderStateEntity.PAID).initial()
        .state(OrderStateEntity.PAID).final()
    )


class SpecialOrderStateEntity(OrderStateEntity):
    """Reuse the parent statuses."""


class SpecialOrderLifecycle(OrderLifecycle, Lifecycle[SpecialOrderStateEntity]):
    """Reuse the parent template with a narrowed state type."""

## Run the example

The following cell performs the operation described above. Its assertions check the result; any expected refusal is caught explicitly.

In [2]:
assert SpecialOrderStateEntity.members == OrderStateEntity.members
assert SpecialOrderStateEntity.by_code("paid") is OrderStateEntity.PAID
print(SpecialOrderStateEntity.codes())
print(SpecialOrderLifecycle("draft").transition("paid").current_state)

['draft', 'paid']
paid


## What the result means

A state subtype with no new members inherits its parent’s catalogue. An axis parameterized by that subtype can therefore refer to those shared statuses. This differs from a dictionary subclass, which owns a separate catalogue.